# Tracekit Bundles

Tracekit records each tool call an agent makes as a signed record, written by a signer the agent does not control, and exports a run as a bundle (`.tkb`) that verifies offline. The `TracekitAdapter` verifies the bundle with Tracekit's verifier before it maps anything.

This notebook uses the sample bundle from the test data: a support agent's six tool calls, with a person answering two held refunds. The metrics below are deterministic, so no AWS account is needed.

#### Install UAEF with the Tracekit extra
`pip install "uaef[tracekit]"`

In [ ]:
from uaef.adapters import get_adapter
from uaef.api import evaluate
from uaef.models import GroundTruth

#### Step 1: Verify and transform the bundle

In [ ]:
bundle = {
    "bundle_path": "../tests/data/tracekit/run.tkb",  # tracekit export --v2 --run RUN -o run.tkb
    "trust_path": "../tests/data/tracekit/trust.json",  # tracekit signer trust -o trust.json
}
trace = get_adapter("tracekit").transform_to_canonical(bundle)

verification = trace.metadata["tracekit"]["verification"]
print("\n".join(verification["summary"]))

#### Step 2: Read the labels
Each tool call has a label in `trace.metadata["tracekit"]["calls"]`: the policy's decision and rule ids, a person's answer to a held call, and whether the call ran.

In [ ]:
calls = trace.metadata["tracekit"]["calls"]
for tool_call, label in zip(trace.tool_calls, calls):
    answer = label.get("approval", {}).get("decision", "")
    ran = "ran" if "result" in label else "not run"
    print(f"{tool_call.name:<15}{label['decision']:<6}{','.join(label['rule_ids']):<10}{answer:<8}{ran}")

#### Step 3: Evaluate against the calls the agent was permitted to make

In [ ]:
permitted = [
    tool_call
    for tool_call, label in zip(trace.tool_calls, calls)
    if label["decision"] in ("allow", "flag")
    or label.get("approval", {}).get("decision") == "approve"
]
result = evaluate(
    trace=trace,
    ground_truth=GroundTruth(expected_tool_calls=permitted),
    metrics=["tool_selection_accuracy", "tool_sequence_correctness"],
)

for dimension in result.dimension_results:
    for metric in dimension.metric_scores:
        print(f"{metric.metric_name}: {metric.score:.2f} ({metric.reasoning})")
print(f"\n{result.overall_score:.2f} on run {trace.metadata['tracekit']['run_id']}: {verification['reason']}")